In [5]:
%pip install lightning torchvision tqdm ipywidgets

Note: you may need to restart the kernel to use updated packages.


# Data

## Dataframes

In [6]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
input_folder = '/kaggle/input/competitions/Kannada-MNIST'

full = pd.read_csv(f"{input_folder}/train.csv")
train, val = train_test_split(full, test_size = 0.2, stratify=full["label"], random_state=7)
test = pd.read_csv(f"{input_folder}/Dig-MNIST.csv")
infer = pd.read_csv(f"{input_folder}/test.csv")

In [7]:
train.columns

Index(['label', 'pixel0', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5',
       'pixel6', 'pixel7', 'pixel8',
       ...
       'pixel774', 'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779',
       'pixel780', 'pixel781', 'pixel782', 'pixel783'],
      dtype='object', length=785)

## Datasets

In [8]:
from torch.utils.data import Dataset
import torch

class KannadaMnist(Dataset):
    def __init__(self, df, inference = False, transform = None):
        super().__init__()
        self.X, self.y = self._prepare_df(df, inference)
        self.transform = transform
        
    @staticmethod
    def _prepare_df(df, inference):
        pixel_cols = [f"pixel{i}" for i in range(784)]
        if inference:
            y = torch.tensor(df["id"].values, dtype=torch.long)
        else:
            y = torch.tensor(df["label"].values, dtype=torch.long)
            
        X = torch.tensor(df[pixel_cols].values, dtype=torch.float32).reshape(-1,1,28,28)/255.0
        return X,y
    @property
    def mean_std(self):
        return self.X.mean(), self.X.std()
    def __len__(self):
        return len(self.X)
    def __getitem__(self,idx):
        x, y = self.X[idx], self.y[idx]
        if self.transform is not None:
            x = self.transform(x)
        return x,y

train_ds_raw = KannadaMnist(train)
mu, sigma = train_ds_raw.mean_std
mu, sigma

(tensor(0.0822), tensor(0.2417))

## Transforms

In [9]:
from torchvision.transforms import v2

train_transform = v2.Compose([
    v2.RandomApply([
        v2.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.92, 1.08)),
    ], p=0.25),
    v2.RandomApply([
        v2.RandomRotation(5),
    ], p=0.5),
    v2.Normalize(mean=[mu], std=[sigma]),
])

val_transform = v2.Compose([
    v2.Normalize(mean=[mu], std=[sigma])
])

## Data Module

In [10]:
import lightning as L
from torch.utils.data import DataLoader

class KannadaMnistDataModule(L.LightningDataModule):
    def __init__(self, batch_size = 32, num_workers = 4):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage : str):
        self.train_ds = KannadaMnist(train, transform = train_transform)
        self.val_ds = KannadaMnist(val, transform = val_transform)
        self.test_ds = KannadaMnist(test, transform = val_transform)
        self.infer_ds = KannadaMnist(infer, inference = True, transform = val_transform)

    def train_dataloader(self):
        return DataLoader(dataset = self.train_ds, batch_size = self.batch_size, num_workers = self.num_workers)
    def val_dataloader(self):
        return DataLoader(dataset = self.val_ds, batch_size = self.batch_size, num_workers = self.num_workers)
    def test_dataloader(self):
        return DataLoader(dataset = self.test_ds, batch_size = self.batch_size, num_workers = self.num_workers)
    def predict_dataloader(self):
        return DataLoader(dataset = self.infer_ds, batch_size = self.batch_size, num_workers = self.num_workers)
    
        

# Model

## Residual blocks


In [11]:
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias = False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias = False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            projection = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
            self.shortcut = projection
    
    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + self.shortcut(x)
        out = self.relu(out)
        return out

class Stem(nn.Module):
    def __init__(self, in_channels, out_channels = 64, kernel_size = 3, stride = 1, padding = 1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias = False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)
        return out



## ResNet

In [12]:
class ResNet(nn.Module):
    def __init__(self, num_blocks, num_classes, in_channels):
        super().__init__()
        self.stem = Stem(in_channels = in_channels, out_channels = 64)
        
        self.res_blocks = nn.ModuleList()
        c_in = 64
        c_out = 64
        strides = [1 if i == 0 else 2 for i in range(len(num_blocks))]
        for stage_idx, n in enumerate(num_blocks):
            stride = strides[stage_idx]
            for block_idx in range(n):
                block_stride = stride if block_idx == 0 else 1
                self.res_blocks.append(ResidualBlock(c_in, c_out, stride=block_stride))
                c_in = c_out
            c_out *= 2
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.2)
        self.head = nn.Sequential(
            nn.Linear(c_in, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        out = self.stem(x)

        for block in self.res_blocks:
            out = block(out) # shape: (batch_size, c_in, h, w)

        out = self.avg_pool(out) # shape: (batch_size, c_in, 1, 1)
        out = out.flatten(1) # shape: (batch_size, c_in)
        out = self.dropout(out)
        out = self.head(out)
        return out


## LitModel

In [13]:
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score
)
import torch
class LitResNet(L.LightningModule):
    def __init__(self, model, num_classes, lr = 1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.loss_fn = nn.CrossEntropyLoss()
        metrics = MetricCollection([
            MulticlassAccuracy(num_classes=num_classes),
            MulticlassPrecision(num_classes=num_classes, average="macro"),
            MulticlassRecall(num_classes=num_classes, average="macro"),
            MulticlassF1Score(num_classes=num_classes, average="macro"),
        ])
        self.train_metrics = metrics.clone(prefix="train/")
        self.val_metrics = metrics.clone(prefix="val/")
        self.test_metrics = metrics.clone(prefix="test/")
    def forward(self,x):
        return self.model(x)
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        output = self.train_metrics(logits, y)
        self.log("train/loss", loss, prog_bar = True)
        self.log_dict(output, on_step=False, on_epoch=True, prog_bar = True)
        return loss
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        output = self.val_metrics(logits, y)
        self.log("val/loss", loss, prog_bar = True)
        self.log_dict(output, on_step=False, on_epoch=True, prog_bar = True)
        return loss
    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        output = self.train_metrics(logits, y)
        self.log("test/loss", loss)
        self.log_dict(output, on_step=False, on_epoch=True)
        return loss
    def predict_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        preds = logits.argmax(dim=1)
        return preds

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.parameters(), lr = self.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)
        lr_scheduler = {
            "scheduler" : scheduler,
            "monitor" : "val/loss",
            "interval" : "epoch"
        }
        return {
            "optimizer" : optimizer,
            "lr_scheduler" : lr_scheduler
        }

# Training

## Callback

In [14]:
from lightning.pytorch.callbacks import Callback
from torchvision.utils import make_grid
from IPython.display import display, Image
import matplotlib.pyplot as plt
import io


class LiveBatchPreview(Callback):
    def __init__(self, mean = mu, std = sigma, every_n_batches=200, n_images=5):
        self.every_n_batches = every_n_batches
        self.n_images = n_images
        self.to_displayable = v2.Compose([
            v2.Normalize(mean=[-mean / std], std=[1 / std]),
            v2.Lambda(lambda x: x.clamp(0, 1)),              
            v2.ToPILImage()
        ])
        self._display_id = "live_batch_preview"
        self._initialized = False

    def _fig_to_image(self, fig):
        buf = io.BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        return Image(data=buf.read())

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        if batch_idx % self.every_n_batches != 0:
            return

        x, y = batch
        n = min(self.n_images, x.shape[0])

        fig, axes = plt.subplots(1, n, figsize=(1.6 * n, 1.6))
        for i in range(n):
            img = self.to_displayable(x[i].detach().cpu())
            axes[i].imshow(img, cmap="gray", vmin=0, vmax=1)
            axes[i].set_title(str(y[i].item()), fontsize=9)
            axes[i].axis("off")
        fig.suptitle(f"epoch {trainer.current_epoch}  batch {batch_idx}", fontsize=10)

        img_widget = self._fig_to_image(fig)
        plt.close(fig)

        if not self._initialized:
            display(img_widget, display_id=self._display_id)
            self._initialized = True
        else:
            display(img_widget, display_id=self._display_id, update=True)

## Trainer

In [15]:
model = ResNet(num_blocks = [3,4,6,3], num_classes = 10 , in_channels = 1)
lit_model = LitResNet(model = model, num_classes = 10)
dm = KannadaMnistDataModule(batch_size = 128)

from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
early_stop = EarlyStopping(
    monitor="val/loss",
    patience=5,
    mode="min",
)
checkpoint_callback = ModelCheckpoint(
    monitor="val/loss",
    mode="min",
    save_top_k=1,
    filename="best-model",
    dirpath="/kaggle/working/checkpoints"
)
EPOCHS = 30
trainer = L.Trainer(max_epochs = EPOCHS, accelerator = "gpu", devices=1, callbacks = [early_stop, checkpoint_callback])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Training loop

In [ ]:
trainer.fit(lit_model,datamodule = dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ ResNet           │ 21.3 M │ train │     0 │
│ 1 │ loss_fn       │ CrossEntropyLoss │      0 │ train │     0 │
│ 2 │ train_metrics │ MetricCollection │      0 │ train │     0 │
│ 3 │ val_metrics   │ MetricCollection │      0 │ train │     0 │
│ 4 │ test_metrics  │ MetricCollection │      0 │ train │     0 │
└───┴───────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 21.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 21.3 M                                                                                               
Total estimated model params size (MB): 85.238                                                                     
Modules in train mode: 148                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

In [ ]:
best_model = LitResNet.load_from_checkpoint(
    checkpoint_callback.best_model_path,
    model=ResNet(num_blocks=[3,4,6,3], num_classes=10, in_channels=1),
    num_classes=10,
)
best_model.eval()

In [ ]:
preds = torch.cat(trainer.predict(best_model, datamodule = dm))

def generate_submission(preds):
    ids = infer["id"]
    cols = ["id", "label"]
    submission = pd.DataFrame(
        {
            "id" : ids,
            "label" : preds.numpy()
        }
    )
    return submission


    

In [ ]:
submission = generate_submission(preds)
submission.to_csv('submission.csv', index = False)

In [ ]:
trainer.validate(best_model, dm)

In [ ]:
trainer.test(best_model, dm)